# 面试题：如何不用现成 Transformer 手写 Encoder-Decoder，并正确做 teacher forcing？

## 面试回答主线

Encoder 使用双向 self-attention 把源序列编码成上下文 memory；Decoder 先做 causal self-attention，再用 cross-attention 让每个目标位置读取 encoder memory，最后映射到词表。标准 block 还包含 LayerNorm、残差与逐位置 FFN。teacher forcing 的 decoder 输入必须是 `[BOS] + target[:-1]`，监督标签是完整 target；若把 target 本身喂给相同位置，模型会直接看到答案，loss 虚假接近零。训练、贪心/beam 解码和 padding mask 必须使用同一 token/position 合同。

## 真实案例：把无序客服槽位规范化为“动作-对象-渠道”

十条输入把动作、对象、渠道以不同顺序写出，目标统一成可供下游 API 使用的三槽格式，再追加 `[EOS]`。数据为教学构造，用来验证 cross-attention 与目标 shift，不代表自然语言泛化。

In [1]:
import math  # 导入平方根以实现注意力缩放和手写 GELU。
import torch  # 导入 PyTorch 以手写完整 Encoder-Decoder 并真实训练。
from torch import nn  # 导入基础模块、参数和 ModuleList。
import torch.nn.functional as F  # 导入多类交叉熵用于序列监督。
torch.set_num_threads(1)  # 小张量教学实验固定单线程以快速复现。
examples = [  # 构造无序槽位到标准动作-对象-渠道序列的映射。
    (["微信", "退款", "查询"], ["查询", "退款", "微信"]),  # 渠道在前的退款查询。
    (["退款", "信用卡", "查询"], ["查询", "退款", "信用卡"]),  # 对象在前的信用卡查询。
    (["取消", "APP", "订单"], ["取消", "订单", "APP"]),  # 动作在前但对象与渠道逆序。
    (["地址", "修改", "APP"], ["修改", "地址", "APP"]),  # 对象在前的地址修改。
    (["物流页", "包裹", "查询"], ["查询", "包裹", "物流页"]),  # 渠道在前的包裹查询。
    (["优惠券", "网页", "查询"], ["查询", "优惠券", "网页"]),  # 对象在前的优惠券查询。
    (["修改", "微信", "手机号"], ["修改", "手机号", "微信"]),  # 动作在前的手机号修改。
    (["订单", "取消", "网页"], ["取消", "订单", "网页"]),  # 对象在前的订单取消。
    (["查询", "退款", "APP"], ["查询", "退款", "APP"]),  # 已经符合标准顺序的退款查询。
    (["修改", "地址", "网页"], ["修改", "地址", "网页"]),  # 已经符合标准顺序的地址修改。
]  # 结束十条槽位规范化样本。
print("源槽位                     目标槽位")  # 输出真实案例输入表标题。
for source, target in examples:  # 逐条展示无序输入和标准输出。
    print(f"{' | '.join(source):<26} -> {' | '.join(target)} | [EOS]")  # 输出源顺序与完整目标协议。

源槽位                     目标槽位
微信 | 退款 | 查询               -> 查询 | 退款 | 微信 | [EOS]
退款 | 信用卡 | 查询              -> 查询 | 退款 | 信用卡 | [EOS]
取消 | APP | 订单              -> 取消 | 订单 | APP | [EOS]
地址 | 修改 | APP              -> 修改 | 地址 | APP | [EOS]
物流页 | 包裹 | 查询              -> 查询 | 包裹 | 物流页 | [EOS]
优惠券 | 网页 | 查询              -> 查询 | 优惠券 | 网页 | [EOS]
修改 | 微信 | 手机号              -> 修改 | 手机号 | 微信 | [EOS]
订单 | 取消 | 网页               -> 取消 | 订单 | 网页 | [EOS]
查询 | 退款 | APP              -> 查询 | 退款 | APP | [EOS]
修改 | 地址 | 网页               -> 修改 | 地址 | 网页 | [EOS]


## Baseline（基线）：直接复制源顺序

copy baseline 保留所有信息，却不理解槽位角色。只有原本已经是“动作-对象-渠道”的输入能 exact match，其他样本虽然 token 集合正确，API 参数顺序仍错误。

In [2]:
baseline_predictions = [source + ["[EOS]"] for source, _ in examples]  # 直接复制源 token 并追加结束符。
gold_sequences = [target + ["[EOS]"] for _, target in examples]  # 构造完整标准目标序列。
baseline_matches = [prediction == gold for prediction, gold in zip(baseline_predictions, gold_sequences)]  # 逐样本检查 exact match。
baseline_accuracy = sum(baseline_matches) / len(baseline_matches)  # 计算严格序列准确率。
print("样本  copy输出                              gold                                  exact")  # 输出基线逐样本结果表标题。
for index, (prediction, gold, matched) in enumerate(zip(baseline_predictions, gold_sequences, baseline_matches), start=1):  # 遍历全部复制结果。
    print(f"{index:>4}  {' '.join(prediction):<38} {' '.join(gold):<38} {matched}")  # 输出预测、标准序列和严格匹配结果。
print(f"直接复制 exact match：{baseline_accuracy:.1%}")  # 输出后续 Encoder-Decoder 的同数据基线。

样本  copy输出                              gold                                  exact
   1  微信 退款 查询 [EOS]                         查询 退款 微信 [EOS]                         False
   2  退款 信用卡 查询 [EOS]                        查询 退款 信用卡 [EOS]                        False
   3  取消 APP 订单 [EOS]                        取消 订单 APP [EOS]                        False
   4  地址 修改 APP [EOS]                        修改 地址 APP [EOS]                        False
   5  物流页 包裹 查询 [EOS]                        查询 包裹 物流页 [EOS]                        False
   6  优惠券 网页 查询 [EOS]                        查询 优惠券 网页 [EOS]                        False
   7  修改 微信 手机号 [EOS]                        修改 手机号 微信 [EOS]                        False
   8  订单 取消 网页 [EOS]                         取消 订单 网页 [EOS]                         False
   9  查询 退款 APP [EOS]                        查询 退款 APP [EOS]                        True
  10  修改 地址 网页 [EOS]                         修改 地址 网页 [EOS]                         True
直接复制 exact match：2

## 核心实现一：手写 LayerNorm、FFN 与通用多头 Attention

同一个 Attention 类支持 encoder self-attention、decoder causal self-attention 和 cross-attention：Q 来自 `query_hidden`，K/V 来自 `key_value_hidden`。没有调用 `nn.Transformer` 或 `nn.MultiheadAttention`。

In [3]:
class ManualLayerNorm(nn.Module):  # 定义沿隐藏维度归一化的可学习层。
    def __init__(self, dimension, epsilon=1e-5):  # 初始化缩放、偏置和稳定常数。
        super().__init__()  # 注册基础模块状态。
        self.scale = nn.Parameter(torch.ones(dimension))  # 创建逐维可学习缩放。
        self.bias = nn.Parameter(torch.zeros(dimension))  # 创建逐维可学习偏置。
        self.epsilon = epsilon  # 保存防止除零的稳定常数。
    def forward(self, hidden):  # 对每个 token 独立执行归一化。
        mean = hidden.mean(dim=-1, keepdim=True)  # 计算隐藏维度均值。
        variance = ((hidden - mean) ** 2).mean(dim=-1, keepdim=True)  # 计算总体方差。
        normalized = (hidden - mean) / torch.sqrt(variance + self.epsilon)  # 得到近似零均值单位方差表示。
        return normalized * self.scale + self.bias  # 应用可学习仿射变换。
def manual_gelu(hidden):  # 使用 erf 公式定义逐元素 GELU。
    return 0.5 * hidden * (1.0 + torch.erf(hidden / math.sqrt(2.0)))  # 返回平滑非线性激活。
class FeedForward(nn.Module):  # 定义逐 token 两层前馈子网络。
    def __init__(self, model_dim, hidden_dim):  # 初始化扩展与压缩投影。
        super().__init__()  # 注册基础模块状态。
        self.first_weight = nn.Parameter(torch.randn(model_dim, hidden_dim) * 0.08)  # 创建隐藏到扩展维参数。
        self.first_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建扩展层偏置。
        self.second_weight = nn.Parameter(torch.randn(hidden_dim, model_dim) * 0.08)  # 创建扩展维回模型维参数。
        self.second_bias = nn.Parameter(torch.zeros(model_dim))  # 创建输出偏置。
    def forward(self, hidden):  # 执行线性扩展、GELU 和线性压缩。
        expanded = hidden @ self.first_weight + self.first_bias  # 把每个 token 投影到更宽空间。
        return manual_gelu(expanded) @ self.second_weight + self.second_bias  # 激活后恢复模型维度。
class MultiHeadAttention(nn.Module):  # 定义可用于 self 与 cross 的通用多头 Attention。
    def __init__(self, model_dim, head_count):  # 初始化 Q/K/V/O 四组参数。
        super().__init__()  # 注册基础模块状态。
        self.model_dim = model_dim  # 保存总隐藏维度。
        self.head_count = head_count  # 保存并行 head 数量。
        self.head_dim = model_dim // head_count  # 计算每个 head 的子空间维度。
        self.query_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.08)  # 创建查询投影参数。
        self.key_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.08)  # 创建键投影参数。
        self.value_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.08)  # 创建值投影参数。
        self.output_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.08)  # 创建多头合并投影参数。
    def split_heads(self, tensor):  # 把最后隐藏维拆分为 head 与 head_dim。
        batch_size, sequence_length, _ = tensor.shape  # 读取批量与序列长度。
        return tensor.reshape(batch_size, sequence_length, self.head_count, self.head_dim).transpose(1, 2)  # 返回 batch、head、length、head_dim 顺序。
    def forward(self, query_hidden, key_value_hidden, causal=False):  # 计算 self 或 cross scaled dot-product attention。
        queries = self.split_heads(query_hidden @ self.query_weight)  # 从 query 输入投影并拆分 Q。
        keys = self.split_heads(key_value_hidden @ self.key_weight)  # 从 memory 或自身投影并拆分 K。
        values = self.split_heads(key_value_hidden @ self.value_weight)  # 从 memory 或自身投影并拆分 V。
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算缩放后的完整 QK 分数。
        if causal:  # decoder self-attention 需要禁止读取未来目标 token。
            query_length = query_hidden.shape[1]  # 读取 decoder 当前目标长度。
            key_length = key_value_hidden.shape[1]  # 读取自注意力键长度。
            future = torch.triu(torch.ones(query_length, key_length, dtype=torch.bool), diagonal=1)  # 创建严格上三角未来 mask。
            scores = scores.masked_fill(future[None, None, :, :], torch.finfo(scores.dtype).min)  # 向 batch 与 head 广播未来屏蔽。
        weights = torch.softmax(scores, dim=-1)  # 沿 key 维归一化为注意力概率。
        head_outputs = weights @ values  # 在每个 head 内汇聚 V。
        merged = head_outputs.transpose(1, 2).reshape(query_hidden.shape[0], query_hidden.shape[1], self.model_dim)  # 合并并恢复总隐藏维度。
        return merged @ self.output_weight, weights  # 返回输出投影后的上下文与完整权重。
torch.manual_seed(5)  # 固定机制预览的参数与输入。
preview_attention = MultiHeadAttention(8, 2)  # 创建八维两头通用 Attention。
preview_query = torch.randn(2, 4, 8)  # 构造四长度 decoder 查询张量。
preview_memory = torch.randn(2, 3, 8)  # 构造三长度 encoder memory 张量。
preview_cross, preview_cross_weights = preview_attention(preview_query, preview_memory, causal=False)  # 执行真实 cross-attention 前向。
print("cross Q/memory/output/weights 形状：", preview_query.shape, preview_memory.shape, preview_cross.shape, preview_cross_weights.shape)  # 输出跨注意力张量合同。
print("第一样本 head0 第一个目标位置权重：", [round(float(value), 4) for value in preview_cross_weights[0, 0, 0]])  # 输出真实三源位置概率。

cross Q/memory/output/weights 形状： torch.Size([2, 4, 8]) torch.Size([2, 3, 8]) torch.Size([2, 4, 8]) torch.Size([2, 2, 4, 3])
第一样本 head0 第一个目标位置权重： [0.3303, 0.3401, 0.3295]


## 核心实现二：Encoder block、Decoder block 与完整模型

Encoder 是 `self-attn + FFN`，Decoder 是 `causal self-attn + cross-attn + FFN`，每个子层都使用 Pre-LN 和残差。训练输入严格右移：decoder 输入 `[BOS, y₀, y₁, y₂]`，标签 `[y₀, y₁, y₂, EOS]`。

In [4]:
class EncoderBlock(nn.Module):  # 定义一个 Pre-LN Transformer encoder block。
    def __init__(self, model_dim, head_count):  # 初始化归一化、自注意力和 FFN。
        super().__init__()  # 注册基础模块状态。
        self.attention_norm = ManualLayerNorm(model_dim)  # 创建 encoder self-attention 前归一化。
        self.attention = MultiHeadAttention(model_dim, head_count)  # 创建双向 self-attention。
        self.feedforward_norm = ManualLayerNorm(model_dim)  # 创建 FFN 前归一化。
        self.feedforward = FeedForward(model_dim, model_dim * 3)  # 创建三倍扩展的逐 token FFN。
    def forward(self, hidden):  # 执行 encoder 的两个残差子层。
        normalized = self.attention_norm(hidden)  # 归一化 self-attention 输入。
        attention_output, weights = self.attention(normalized, normalized, causal=False)  # 执行双向 encoder self-attention。
        hidden = hidden + attention_output  # 第一条残差加入跨源位置上下文。
        hidden = hidden + self.feedforward(self.feedforward_norm(hidden))  # 第二条残差加入逐位置非线性特征。
        return hidden, weights  # 返回 encoder memory 与自注意力权重。
class DecoderBlock(nn.Module):  # 定义含 masked self 与 cross 的 Pre-LN decoder block。
    def __init__(self, model_dim, head_count):  # 初始化三组归一化与子层。
        super().__init__()  # 注册基础模块状态。
        self.self_norm = ManualLayerNorm(model_dim)  # 创建 masked self-attention 前归一化。
        self.self_attention = MultiHeadAttention(model_dim, head_count)  # 创建目标侧 self-attention。
        self.cross_norm = ManualLayerNorm(model_dim)  # 创建 cross-attention 查询归一化。
        self.cross_attention = MultiHeadAttention(model_dim, head_count)  # 创建读取 encoder memory 的注意力。
        self.feedforward_norm = ManualLayerNorm(model_dim)  # 创建 decoder FFN 前归一化。
        self.feedforward = FeedForward(model_dim, model_dim * 3)  # 创建三倍扩展的 decoder FFN。
    def forward(self, hidden, memory):  # 执行 decoder 三个残差子层。
        normalized = self.self_norm(hidden)  # 归一化目标侧自注意力输入。
        self_output, self_weights = self.self_attention(normalized, normalized, causal=True)  # 执行严格 causal self-attention。
        hidden = hidden + self_output  # 残差加入已生成目标上下文。
        cross_query = self.cross_norm(hidden)  # 归一化 cross-attention 查询表示。
        cross_output, cross_weights = self.cross_attention(cross_query, memory, causal=False)  # 让每个目标位置读取全部 encoder memory。
        hidden = hidden + cross_output  # 残差加入源序列信息。
        hidden = hidden + self.feedforward(self.feedforward_norm(hidden))  # 残差加入逐位置非线性特征。
        return hidden, self_weights, cross_weights  # 返回 decoder 表示与两类注意力。
class TinySeq2SeqTransformer(nn.Module):  # 定义从源 token 到目标词表 logits 的完整 Encoder-Decoder。
    def __init__(self, vocabulary_size, model_dim, head_count, max_source, max_target):  # 初始化共享 embedding、位置表、blocks 与输出头。
        super().__init__()  # 注册基础模块状态。
        self.token_weight = nn.Parameter(torch.randn(vocabulary_size, model_dim) * 0.10)  # 创建源目标共享 token embedding。
        self.source_position = nn.Parameter(torch.randn(max_source, model_dim) * 0.05)  # 创建源序列绝对位置表。
        self.target_position = nn.Parameter(torch.randn(max_target, model_dim) * 0.05)  # 创建目标序列绝对位置表。
        self.encoder = EncoderBlock(model_dim, head_count)  # 创建一个自研 encoder block。
        self.decoder = DecoderBlock(model_dim, head_count)  # 创建一个自研 decoder block。
        self.final_norm = ManualLayerNorm(model_dim)  # 创建词表投影前最终归一化。
        self.output_weight = nn.Parameter(torch.randn(vocabulary_size, model_dim) * 0.10)  # 创建目标词表输出参数。
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))  # 创建目标词表偏置。
    def encode(self, source_ids):  # 把源序列编码为可供 cross-attention 读取的 memory。
        source_length = source_ids.shape[1]  # 读取当前源长度。
        source_hidden = self.token_weight[source_ids] + self.source_position[:source_length].unsqueeze(0)  # 相加源 token 与位置表示。
        memory, encoder_weights = self.encoder(source_hidden)  # 通过双向 encoder block。
        return memory, encoder_weights  # 返回源 memory 和可解释自注意力。
    def decode(self, decoder_ids, memory):  # 在给定 encoder memory 上计算目标 logits。
        target_length = decoder_ids.shape[1]  # 读取当前目标前缀长度。
        target_hidden = self.token_weight[decoder_ids] + self.target_position[:target_length].unsqueeze(0)  # 相加目标 token 与位置表示。
        decoded, self_weights, cross_weights = self.decoder(target_hidden, memory)  # 执行 masked self、cross 与 FFN。
        normalized = self.final_norm(decoded)  # 对最终目标表示执行手写归一化。
        logits = normalized @ self.output_weight.transpose(0, 1) + self.output_bias  # 映射每个目标位置到完整词表。
        return logits, self_weights, cross_weights  # 返回词表分数和两类目标注意力。
    def forward(self, source_ids, decoder_ids):  # 执行完整 Encoder-Decoder teacher-forcing 前向。
        memory, encoder_weights = self.encode(source_ids)  # 编码全部源槽位。
        logits, self_weights, cross_weights = self.decode(decoder_ids, memory)  # 用右移目标前缀生成词表分数。
        return logits, encoder_weights, self_weights, cross_weights  # 返回训练输出和三类注意力。
special_tokens = ["[BOS]", "[EOS]"]  # 定义目标序列协议所需特殊 token。
vocabulary = sorted({token for source, target in examples for token in source + target} | set(special_tokens))  # 收集源、目标和特殊 token 的统一词表。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 创建稳定 token id 映射。
source_ids = torch.tensor([[token_to_id[token] for token in source] for source, _ in examples], dtype=torch.long)  # 转换十条三长度源序列。
target_ids = torch.tensor([[token_to_id[token] for token in target + ["[EOS]"]] for _, target in examples], dtype=torch.long)  # 转换四长度监督目标。
decoder_input_ids = torch.tensor([[token_to_id[token] for token in ["[BOS]"] + target] for _, target in examples], dtype=torch.long)  # 构造严格右移的 decoder 输入。
torch.manual_seed(163)  # 固定 Encoder-Decoder 参数初始化。
seq2seq_model = TinySeq2SeqTransformer(len(vocabulary), 20, 2, 3, 5)  # 创建二十维两头单层 Encoder-Decoder。
training_trace = []  # 保存代表轮次的 token loss 与 exact match。
for step in range(1401):  # 执行真实 teacher-forcing 前向、反向和手动 SGD。
    logits, encoder_weights, self_weights, cross_weights = seq2seq_model(source_ids, decoder_input_ids)  # 计算全部目标位置词表分数。
    loss = F.cross_entropy(logits.reshape(-1, len(vocabulary)), target_ids.reshape(-1))  # 对四个目标位置计算平均交叉熵。
    loss.backward()  # 对共享 embedding、encoder、decoder 和词表头执行真实反向传播。
    with torch.no_grad():  # 手动参数更新不进入下一轮计算图。
        for parameter in seq2seq_model.parameters():  # 遍历完整 Encoder-Decoder 的全部参数。
            parameter -= 0.05 * parameter.grad  # 使用固定学习率执行 SGD 更新。
            parameter.grad.zero_()  # 清空本轮梯度避免错误累积。
    if step in {0, 20, 100, 300, 700, 1400}:  # 记录能够说明优化过程的代表轮次。
        token_predictions = logits.detach().argmax(dim=2)  # 取得 teacher-forcing 下每位置最高分 token。
        exact = float((token_predictions == target_ids).all(dim=1).to(torch.float32).mean())  # 计算严格序列匹配率。
        training_trace.append((step, float(loss.detach()), exact))  # 保存轮次、损失和 exact match。
print("轮次 | token交叉熵 | teacher-forcing exact")  # 输出真实训练轨迹表标题。
for step, loss_value, exact in training_trace:  # 逐个展示代表训练状态。
    print(f"{step:>4} | {loss_value:>11.4f} | {exact:>21.1%}")  # 输出轮次、loss 和严格序列准确率。

轮次 | token交叉熵 | teacher-forcing exact
   0 |      2.8007 |                  0.0%
  20 |      1.2393 |                 10.0%
 100 |      0.3413 |                 70.0%
 300 |      0.0597 |                100.0%
 700 |      0.0107 |                100.0%
1400 |      0.0042 |                100.0%


## 贪心自回归解码与 cross-attention 结果表

评估不再喂 gold 前缀：每次从 `[BOS]` 开始，只把模型上一步预测追加到 decoder 输入。下面比较 copy baseline 与真正 greedy exact match，并打印第一条样本各输出位置对三个源槽位的 cross-attention。

In [5]:
def greedy_decode(model, source_row, max_steps=4):  # 从 `[BOS]` 开始逐 token 自回归生成目标序列。
    decoder_ids = torch.tensor([[token_to_id["[BOS]"]]], dtype=torch.long)  # 初始化只含开始符的目标前缀。
    generated = []  # 创建列表保存可读预测 token。
    cross_rows = []  # 保存每个生成位置的第零头 cross-attention。
    with torch.no_grad():  # 推理解码不需要梯度图。
        memory, encoder_weights = model.encode(source_row)  # 源序列只编码一次得到 memory。
        for step in range(max_steps):  # 最多生成三个槽位和一个结束符。
            logits, self_weights, cross_weights = model.decode(decoder_ids, memory)  # 对当前目标前缀执行 causal decoder。
            next_id = int(logits[0, -1].argmax())  # 取得最后位置最高概率词表编号。
            next_token = vocabulary[next_id]  # 把编号还原为可读 token。
            generated.append(next_token)  # 按生成顺序保存当前预测。
            cross_rows.append(cross_weights[0, 0, -1].detach())  # 保存当前输出位置对源三 token 的注意力。
            decoder_ids = torch.cat([decoder_ids, torch.tensor([[next_id]], dtype=torch.long)], dim=1)  # 把当前预测追加为下一步输入。
            if next_token == "[EOS]":  # 生成结束符时提前停止。
                break  # 结束当前样本的自回归循环。
    return generated, cross_rows  # 返回完整预测序列和逐步 cross-attention。
greedy_predictions = []  # 保存十条样本的自由运行预测。
all_cross_rows = []  # 保存各样本逐步 cross-attention。
for row_index in range(len(examples)):  # 逐源样本执行独立贪心解码。
    prediction, cross_rows = greedy_decode(seq2seq_model, source_ids[row_index : row_index + 1])  # 生成当前标准槽位序列。
    greedy_predictions.append(prediction)  # 保存当前完整预测。
    all_cross_rows.append(cross_rows)  # 保存当前跨注意力轨迹。
greedy_matches = [prediction == gold for prediction, gold in zip(greedy_predictions, gold_sequences)]  # 逐样本检查严格序列一致。
greedy_accuracy = sum(greedy_matches) / len(greedy_matches)  # 计算自由运行 exact match。
print("样本  source                    copy exact  greedy预测                              greedy exact")  # 输出逐样本结果表标题。
for index, ((source, target), baseline_match, prediction, matched) in enumerate(zip(examples, baseline_matches, greedy_predictions, greedy_matches), start=1):  # 遍历完整对照结果。
    print(f"{index:>4}  {' '.join(source):<24} {str(baseline_match):<10} {' '.join(prediction):<38} {matched}")  # 输出源、基线和真实解码结果。
print(f"exact match 对照：copy={baseline_accuracy:.1%}，手写 Encoder-Decoder={greedy_accuracy:.1%}")  # 输出相同数据和指标下的核心对照。
print("样本1 cross-attention（行=生成位置，列=三个源槽位）：")  # 输出焦点样本跨注意力矩阵标题。
for token, row in zip(greedy_predictions[0], all_cross_rows[0]):  # 对齐焦点预测 token 与其源注意力。
    print(token, "->", [round(float(value), 4) for value in row])  # 输出每个生成位置读取三个源 token 的概率。

样本  source                    copy exact  greedy预测                              greedy exact
   1  微信 退款 查询                 False      查询 退款 微信 [EOS]                         True
   2  退款 信用卡 查询                False      查询 退款 信用卡 [EOS]                        True
   3  取消 APP 订单                False      取消 订单 APP [EOS]                        True
   4  地址 修改 APP                False      修改 地址 APP [EOS]                        True
   5  物流页 包裹 查询                False      查询 包裹 物流页 [EOS]                        True
   6  优惠券 网页 查询                False      查询 优惠券 网页 [EOS]                        True
   7  修改 微信 手机号                False      修改 手机号 微信 [EOS]                        True
   8  订单 取消 网页                 False      取消 订单 网页 [EOS]                         True
   9  查询 退款 APP                True       查询 退款 APP [EOS]                        True
  10  修改 地址 网页                 True       修改 地址 网页 [EOS]                         True
exact match 对照：copy=20.0%，手写 Encoder-Decoder=10

## 结果解读

copy baseline 只能处理原本已规范的输入。自研 Encoder 双向混合源槽位，Decoder 在 causal 目标前缀上通过 cross-attention读取源 memory，真实训练后能输出固定协议。注意力行可用于检查每个输出位置读取了哪些源槽位，但不能替代基于扰动或归因的因果解释。

## 失败案例：decoder 输入没有右移导致标签泄漏

用一个“只复制当前位置输入 id”的探针就能暴露问题：若 decoder 输入直接等于 target，探针无需看源序列就达到近零 loss 和 100% token accuracy；正确右移后，同一个探针无法预测当前位置答案。修复必须发生在数据对齐阶段。

In [6]:
leaky_decoder_inputs = target_ids.clone()  # 构造错误的未右移 decoder 输入，当前位置直接含答案。
correct_decoder_inputs = decoder_input_ids  # 使用训练中实际采用的 `[BOS] + target[:-1]`。
def identity_probe_logits(decoder_ids, vocabulary_size):  # 构造只复制当前位置输入 token 的泄漏探针。
    logits = torch.full((decoder_ids.shape[0], decoder_ids.shape[1], vocabulary_size), -8.0)  # 为所有非输入 token 设置很低分数。
    logits.scatter_(2, decoder_ids.unsqueeze(-1), 8.0)  # 把当前位置输入 token 设置为唯一高分预测。
    return logits  # 返回不读取源序列的词表分数。
leaky_probe_logits = identity_probe_logits(leaky_decoder_inputs, len(vocabulary))  # 在错误未右移数据上运行复制探针。
correct_probe_logits = identity_probe_logits(correct_decoder_inputs, len(vocabulary))  # 在正确右移数据上运行同一探针。
leaky_probe_loss = float(F.cross_entropy(leaky_probe_logits.reshape(-1, len(vocabulary)), target_ids.reshape(-1)))  # 计算标签泄漏下的虚假低 loss。
correct_probe_loss = float(F.cross_entropy(correct_probe_logits.reshape(-1, len(vocabulary)), target_ids.reshape(-1)))  # 计算正确 shift 下的真实高 loss。
leaky_probe_accuracy = float((leaky_probe_logits.argmax(dim=2) == target_ids).to(torch.float32).mean())  # 计算未右移探针 token accuracy。
correct_probe_accuracy = float((correct_probe_logits.argmax(dim=2) == target_ids).to(torch.float32).mean())  # 计算正确右移探针 token accuracy。
print("样本1错误输入：", [vocabulary[index] for index in leaky_decoder_inputs[0].tolist()], "标签：", gold_sequences[0])  # 展示当前位置输入与标签完全相同。
print("样本1正确输入：", [vocabulary[index] for index in correct_decoder_inputs[0].tolist()], "标签：", gold_sequences[0])  # 展示 BOS 和前一目标 token 的正确对齐。
print(f"未右移复制探针：loss={leaky_probe_loss:.6f}，token accuracy={leaky_probe_accuracy:.1%}")  # 输出不读源也能满分的泄漏证据。
print(f"正确右移复制探针：loss={correct_probe_loss:.6f}，token accuracy={correct_probe_accuracy:.1%}")  # 输出修复后探针无法作弊的结果。

样本1错误输入： ['查询', '退款', '微信', '[EOS]'] 标签： ['查询', '退款', '微信', '[EOS]']
样本1正确输入： ['[BOS]', '查询', '退款', '微信'] 标签： ['查询', '退款', '微信', '[EOS]']
未右移复制探针：loss=0.000002，token accuracy=100.0%
正确右移复制探针：loss=16.000002，token accuracy=0.0%


## 生产差距与落地清单

教学模型无 padding、beam、label smoothing、dropout 和 KV cache，只在十条训练样本上验证机制。线上要分别构造 source padding、target padding 与 target causal mask，loss 忽略 PAD，并用独立验证集评估 exact match、槽位 F1、非法协议率和长度分桶。推理需限制 EOS/最大长度、验证 beam 长度惩罚，并测试 encoder memory 复用和 decoder cache 与全量 logits 一致。

## 最小回归测试

断言只保护架构张量、真实优化、自由解码和 shift 合同；类实现、训练轨迹、cross-attention 与泄漏探针才是主要学习证据。

In [7]:
assert preview_cross.shape == preview_query.shape  # 验证 cross-attention 输出保持 query 的批量、长度与隐藏维度。
assert torch.allclose(preview_cross_weights.sum(dim=-1), torch.ones_like(preview_cross_weights.sum(dim=-1)))  # 验证每个 cross-attention 行概率归一化。
assert training_trace[-1][1] < training_trace[0][1]  # 验证自研 Encoder-Decoder 的真实反向传播降低 token 交叉熵。
assert greedy_accuracy > baseline_accuracy  # 验证自由运行规范化优于直接复制源顺序。
assert all(prediction[-1] == "[EOS]" for prediction in greedy_predictions)  # 验证每条自由解码都按协议生成结束符。
assert leaky_probe_accuracy == 1.0 and leaky_probe_loss < 1e-4  # 固化未右移数据可被当前位置复制轻易作弊的反例。
assert correct_probe_accuracy < leaky_probe_accuracy and correct_probe_loss > 1.0  # 验证正确右移后复制探针不再获得虚假好指标。
print("最小回归测试通过：Encoder-Decoder 前向、cross-attention、自由解码与 teacher-forcing shift 均符合预期。")  # 输出完整顺序执行成功的明确结论。

最小回归测试通过：Encoder-Decoder 前向、cross-attention、自由解码与 teacher-forcing shift 均符合预期。
